In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd

os.environ['OPENAI_KEY_SAE'] = '...' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_KEY'])
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8010/v1'  # Optional: local OpenAI-compatible endpoint

from hypothesaes.quickstart import train_sae
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.sae import get_multiple_sae_activations
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, SamplingConfig, sample_percentile_bins, ternary_sample_percentile_bins, ternary_sample_topn
from hypothesaes.evaluation import score_hypotheses
from hypothesaes.annotate import annotate_texts_with_concepts


In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
base_dir = os.path.join(prefix, "data")

EMBEDDER = "nomic-ai/modernbert-embed-base"
ANNOTATOR_MODEL = "openai/gpt-oss-20b"
INTERPRETER_MODEL = "openai/gpt-oss-120b"
INTERPRET_PROMPT_NAME = "interpret-neuron-binary"

USE_ABS_TOPK = True
USE_VALIDATION = True
PERCENTILE_SAMPLE = False
DATASET_NAME = "headlines"

EMBED_CACHE_NAME = f"{DATASET_NAME}_{EMBEDDER}"
ACT_CACHE_NAME = EMBED_CACHE_NAME
if USE_ABS_TOPK:
    ACT_CACHE_NAME += "_abstopk"
    INTERPRET_PROMPT_NAME = "interpret-neuron-ternary"

INTERP_CACHE_NAME = ACT_CACHE_NAME
if not PERCENTILE_SAMPLE:
    INTERP_CACHE_NAME += "_topnsample"

ANN_CACHE_NAME = INTERP_CACHE_NAME
if USE_VALIDATION:
    ANN_CACHE_NAME += "_validation"

In [3]:
print(
    "EMBED_CACHE_NAME =", EMBED_CACHE_NAME, "\n",
    "ACT_CACHE_NAME =", ACT_CACHE_NAME, "\n",
    "INTERP_CACHE_NAME =", INTERP_CACHE_NAME, "\n",
    "ANN_CACHE_NAME =", ANN_CACHE_NAME, "\n",
    "INTERPRET_PROMPT_NAME =", INTERPRET_PROMPT_NAME,
)

EMBED_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base 
 ACT_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk 
 INTERP_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk_topnsample 
 ANN_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk_topnsample_validation 
 INTERPRET_PROMPT_NAME = interpret-neuron-ternary


In [4]:
label_col = "label_pairwise"
train_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "train-8.8K.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "val-1.0K.json"), lines=True)

train_labels = train_df[label_col].values

unique_train_texts = np.unique(train_df["headline_A"].tolist() + train_df["headline_B"].tolist()).tolist()
unique_val_texts = np.unique(val_df["headline_A"].tolist() + val_df["headline_B"].tolist()).tolist()

text2embedding = get_local_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=EMBED_CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in unique_train_texts])
val_embeddings = np.stack([text2embedding[text] for text in unique_val_texts])

Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 16468 embeddings in 0.0s


In [5]:
checkpoint_dir = os.path.join(prefix, f"checkpoints/{ACT_CACHE_NAME}")
sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)
sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)

sae_list = [sae_256_8, sae_32_4]

unique_train_activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, train_embeddings, return_neuron_source_info=True)


Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=256_K=8.pt onto device cuda
Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=32_K=4.pt onto device cuda


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:

headline_A_train = np.stack([text2embedding[text] for text in train_df["headline_A"].tolist()])
headline_B_train = np.stack([text2embedding[text] for text in train_df["headline_B"].tolist()])

Z_A = get_multiple_sae_activations(sae_list, headline_A_train)
Z_B = get_multiple_sae_activations(sae_list, headline_B_train)
Z_delta = Z_A - Z_B

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
selection_cache_dir = os.path.join(prefix, "selection_cache", ACT_CACHE_NAME)
os.makedirs(selection_cache_dir, exist_ok=True)
selected_neurons, scores = select_neurons(
    activations=Z_delta,
    target=train_labels,
    n_select=20,
    method="lasso",
    classification=True,
    verbose=True,
    cache_dir=selection_cache_dir
)

In [8]:
import importlib
import hypothesaes.interpret_neurons
importlib.reload(hypothesaes.interpret_neurons)
from hypothesaes.interpret_neurons import NeuronInterpreter, ternary_sample_percentile_bins, ternary_sample_topn

In [9]:
interpreter = NeuronInterpreter(
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    n_workers_interpretation=4,
    n_workers_annotation=100,
    cache_name=INTERP_CACHE_NAME,
)

n_examples = 20
high_percentile = (80, 100)
n_candidates = 5
annotation_kwargs = {}

if USE_ABS_TOPK:
    if PERCENTILE_SAMPLE:
        interpretation_sampling_config = SamplingConfig(
            function=ternary_sample_percentile_bins,
            n_examples=n_examples,
            sampling_kwargs={
                "high_abs_percentile": high_percentile
            },
        )
    else:
        interpretation_sampling_config = SamplingConfig(
            function=ternary_sample_topn,
            n_examples=n_examples,
        )
else:
    interpretation_sampling_config = SamplingConfig(
        function=sample_percentile_bins,
        n_examples=n_examples,
        sampling_kwargs={
            "high_percentile": high_percentile
        },
    )

interpret_config = InterpretConfig(
    sampling=interpretation_sampling_config,
    n_candidates=n_candidates,
    interpretation_prompt_name=INTERPRET_PROMPT_NAME,
    task_specific_instructions="""All of the texts are digital news headlines.
Features should describe specific characteristics in the headline texts. Example features may include:
- "explicitly mentions climate change or another environmental issue"
- "asks the reader a rhetorical question\"""",
)

In [10]:
interpretations = interpreter.interpret_neurons(
    texts=unique_train_texts,
    activations=unique_train_activations,
    neuron_indices=selected_neurons,
    config=interpret_config,
)

Generating interpretations:   0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
scoring_config = ScoringConfig(
    n_examples=200,
    sampling_function=interpretation_sampling_config.function,
    sampling_kwargs=interpretation_sampling_config.sampling_kwargs
)
score_func = interpreter.ternary_score_interpretations if USE_ABS_TOPK else interpreter.score_interpretations
# score_func = interpreter.score_interpretations

In [12]:
INTERP_CACHE_NAME

'headlines_nomic-ai/modernbert-embed-base_abstopk_topnsample'

In [14]:
all_metrics = score_func(
    texts=unique_train_texts,
    activations=unique_train_activations,
    interpretations=interpretations,
    config=scoring_config,
)

[WARNING] Only 62 < 66 negative examples available for 140
[WARNING] Only 62 < 66 neutral examples available for 140
[WARNING] Only 13 < 66 positive examples available for 62
[WARNING] Only 31 < 66 negative examples available for 210
[WARNING] Only 31 < 66 neutral examples available for 210
[WARNING] Only 7 < 66 negative examples available for 243
[WARNING] Only 7 < 66 neutral examples available for 243
[WARNING] Only 27 < 66 positive examples available for 65
[WARNING] Only 27 < 66 positive examples available for 81
[WARNING] Only 34 < 66 negative examples available for 286
[WARNING] Only 34 < 66 neutral examples available for 286
Found 18495 cached items; annotating 0 uncached items


/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [16]:
results = []
for neuron_idx, fidelity_scores in all_metrics.items():
    best_interp = max(interpretations[neuron_idx], key=lambda interp: fidelity_scores[interp]['f1'])
    results.append({
        'neuron_idx': neuron_idx,
        'source_sae': neuron_source_sae_info[neuron_idx],
        'target_lasso_coef': scores[selected_neurons.index(neuron_idx)],
        'interpretation': best_interp,
        'f1_fidelity_score': fidelity_scores[best_interp]['f1'],
    })

In [18]:
hypothesis_df = pd.DataFrame(results).set_index('neuron_idx')
pd.set_option('display.max_colwidth', None)
display(hypothesis_df.sort_values(by='target_lasso_coef', ascending=False))
pd.reset_option('display.max_colwidth')

,source_sae,target_lasso_coef,interpretation,f1_fidelity_score
neuron_idx,,,,
257,"(SAE_1, 32, 4)",0.158348,Mentions a video (positive) or mentions a book/novel/fictional work (negative),0.758338
262,"(SAE_1, 32, 4)",0.109705,Mentions a marginalized identity or social injustice (positive) or mentions an environmental/climate issue (negative),0.668946
140,"(SAE_0, 256, 8)",0.069642,"Mentions photographic media (photos/images) in positive samples versus mentions video/broadcast media (videos, TV, watching) in negative samples",0.612069
41,"(SAE_0, 256, 8)",0.050278,"Contains a strongly valenced adjective: negative adjectives (e.g., “Worst”, “Terrible”, “Disturbing”) in positive samples; positive adjectives (e.g., “Funny”, “Beautiful”, “Well‑Connected”) in negative samples",0.517170
46,"(SAE_0, 256, 8)",0.039174,"Contains a number written as a word (e.g., “One”) in positive samples or contains a numeric digit (e.g., “7”, “49”) in negative samples",0.453088
274,"(SAE_1, 32, 4)",0.020018,"Mentions a speech‑related term such as “Speech” or “TED Talk” (positive) or mentions a visual‑media term such as “Oscars”, “movie”, or “Tabloid” (negative)",0.656542
139,"(SAE_0, 256, 8)",0.017634,"References a protected demographic group (e.g., race in positive samples, gender in negative samples)",0.402299
260,"(SAE_1, 32, 4)",0.016116,"Contains a numeric token (e.g., 51, 5, 50+)",0.514337
62,"(SAE_0, 256, 8)",0.015230,"Mentions a controversial social issue (e.g., LGBTQ+ rights, consent, sexual violence) in positive samples or mentions a controversial environmental/energy issue (e.g., oil spills, pipelines) in negative samples",0.671395


In [30]:
def show_examples(neuron_idx):
    n_ex = 5
    print(hypothesis_df.loc[neuron_idx].interpretation)
    neuron_acts = unique_train_activations[:, neuron_idx]
    sort_inds = np.argsort(neuron_acts)
    def print_example(ind):
        print(f"{neuron_acts[ind]:.4f} : {unique_train_texts[ind]}")
    print("POSITIVE")
    print("-----------------")
    print("Activation : Text")
    print("-----------------")
    for i in range(-1, -n_ex-1, -1):
        print_example(sort_inds[i])
    print()
    print("NEGATIVE")
    print("-----------------")
    print("Activation : Text")
    print("-----------------")
    for i in range(n_ex):
        print_example(sort_inds[i])

In [31]:
show_examples(139)


References a protected demographic group (e.g., race in positive samples, gender in negative samples)
POSITIVE
-----------------
Activation : Text
-----------------
0.2727 : A black man sits down at a diner to eat. The white patrons are shocked. Then the owner steps up.
0.2467 : A black man sits down at a diner. Stares come from the white patrons. Then the owner speaks.
0.2417 : Here's The Moment A Black Woman Protected A White Man At A KKK Rally
0.2331 : A black man sits down to eat at a diner. Stares from the white patrons. The owner steps up, and ...
0.2275 : Here's A Rarely-Seen-Before Photo Of The Moment A Black Woman Protects A White Man At A KKK Rally

NEGATIVE
-----------------
Activation : Text
-----------------
-0.1373 : His Rap Songs Show A Side Of His Past That We All Need To Keep For The Future
-0.1309 : If You're Not Like Him, Find Someone Who Is And Be A Support For Them. They're Everywhere.
-0.1210 : It's Kinda Like Group Therapy, But For Unbelievably Adorable Little Cl

In [23]:
show_examples(267)


Contains the word 'Men' (positive) or the word 'She'/'Her' (negative)
POSITIVE
--------
0.2325 : What Are We Teaching Young Men About Sex? Maybe Not The Best Stuff.
0.2286 : I Think I Finally Found A Movement That Shows Men The Appreciation They Deserve
0.2186 : Men Could Be Represented  More Effectively With Gender Equality
0.2128 : I Think I Finally Found A Movement That Treats Men With The Respect They Deserve. About Time.
0.2050 : Watch Men Learn What Feminism Means And Then Realize They're Feminists
NEGATIVE
--------
-0.4060 : She Built Herself A Home On The Place She Grew Up. Now Some Powerful People Wanna Throw Her Out.
-0.4018 : She Got All The Fame For Her Amazing Journey, But She Wants You To Know She Didn't Do It Alone
-0.3889 : I Didn't Think She Could Succeed. But Then I Saw How Determined She Is.
-0.3880 : She hated authority until she became boss. Here's how she treats her employees.
-0.3838 : No One Can Question Her Beauty But Why Did She Never Think A Role Model Could 

In [24]:
show_examples(67)

Mentions a singular human referent introduced with a demonstrative (e.g., “This Guy/Man/Veteran”) in positive samples versus mentions the plural pronoun “They” referring to a group in negative samples
POSITIVE
--------
0.1273 : This Guy Looks Like He’s From The Past, But His Truth Is Very Important To The Present
0.1207 : Until The Workplace Gets Out Of The 60s, This Famously Old Fashioned Lady's Staying That Way
0.1155 : Her witty comebacks to these awful comments will have you running for a pen
0.1154 : What Are The Odds That You'll Be Single Forever? This Guy Will Tell You.
0.1139 : This Man Has Gone Viral. And He Won't Stop Until You Do, Too
NEGATIVE
--------
-0.2905 : They Dreamed Of Putting On The Uniform, So Why Were They Forced Out?
-0.2776 : They Were Happy With Everything Until He Confronted Them With What Was Inside
-0.2730 : She Wasn’t OK With What They Wanted Her To Do. I'm Sure You Wouldn't Have Been OK With It, Either.
-0.2691 : Her Friends Wanted A Laugh. What They Got 

## Training Predictor

In [15]:
def display_df(m, evaluation_df):
    print(f"AUC: {m['auroc']:.3f}")
    print(f"Significant hypotheses: {m['Significant'][0]}/{m['Significant'][1]} " 
          f"(p < {m['Significant'][2]:.3e})")
    pd.set_option('display.max_colwidth', None)
    display(evaluation_df.sort_values(by='separation_score', ascending=False).round(4))
    pd.reset_option('display.max_colwidth')
    

In [16]:
Z_preproc = np.where(np.abs(Z_delta) > 1e-6, Z_delta, 0)
Z_preproc = np.sign(Z_preproc)
train_delta_annots = {}
for _, row in hypothesis_df.iterrows():
    hyp, neuron_idx = row[['interpretation', 'neuron_idx']]
    hyp_labels = Z_preproc[:, neuron_idx]
    train_delta_annots[hyp] = hyp_labels

In [17]:
_, _, train_predictor = score_hypotheses(
    hypothesis_annotations=train_delta_annots,
    y_true=train_labels,
    classification=True,
)

Optimization terminated successfully.
         Current function value: 0.641275
         Iterations 5


## Evaluating the Predictor

In [18]:
if USE_VALIDATION:
    holdout_basename = "val-1.0K.json"
else:
    holdout_basename = "holdout-4.4K.json"
holdout_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, holdout_basename), lines=True)

headline_A_holdout = holdout_df["headline_A"].tolist()
headline_B_holdout = holdout_df["headline_B"].tolist()
labels_holdout = holdout_df[label_col].values

hypotheses = hypothesis_df["interpretation"].tolist()

In [19]:
headline_A_annotations = annotate_texts_with_concepts(
    texts=headline_A_holdout,
    concepts=hypotheses,
    cache_name=ANN_CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)

Found 2080 cached items; annotating 18300 uncached items


Annotating:   0%|          | 0/18300 [00:00<?, ?it/s]

In [21]:
ANN_CACHE_NAME

'headlines_nomic-ai/modernbert-embed-base_abstopk_validation'

In [20]:

headline_B_annotations = annotate_texts_with_concepts(
    texts=headline_B_holdout,
    concepts=hypotheses,
    cache_name=ANN_CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)


Found 780 cached items; annotating 19600 uncached items


Annotating:   0%|          | 0/19600 [00:00<?, ?it/s]

In [22]:
eval_delta_annots = {key : headline_A_annotations[key] - headline_B_annotations[key] for key in headline_A_annotations.keys()}
metrics, evaluation_df, _ = score_hypotheses(
    hypothesis_annotations=eval_delta_annots,
    y_true=labels_holdout,
    classification=True,
    predictor=train_predictor
)

In [23]:
display_df(metrics, evaluation_df)

AUC: 0.549
Significant hypotheses: 19/20 (p < 5.000e-03)


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence,significant,auc
Index,,,,,,,,
13,Contains the word “Response”,0.2768,0.1496,0.3942,0.0001,0.0128,True,0.5068
14,"Mentions a specific time duration (e.g., minutes or seconds) in the headline (positive) vs does not mention any time duration (negative)",0.1599,0.0000,0.4266,0.0000,0.0815,True,0.5254
8,Mentions women or ladies (positive) vs mentions he/his (negative),0.1492,0.0000,-0.2347,0.0000,0.1560,True,0.4557
9,"References spoken language (e.g., words, speech, say, hearing) in positive samples versus references visual media/film (e.g., Hollywood, TV, movies, film) in negative samples",0.1030,0.0000,0.2338,0.0000,0.2502,True,0.5454
7,Mentions masculine gender terms such as 'Men' or 'Guys' (positive) versus mentions feminine pronouns like 'She' or 'Her' (negative),0.0996,0.0000,-0.2215,0.0000,0.2041,True,0.4558
5,Conveys a negative sentiment in the headline (positive samples) versus a positive sentiment (negative samples),0.0928,0.0000,0.5031,0.0000,0.3690,True,0.5649
4,"Mentions a specific individual (e.g., Kid, Guy, Lady) in positive samples vs mentions the collective pronoun ‘They’ in negative samples",0.0695,0.0000,-0.3408,0.0000,0.2640,True,0.4601
11,Contains a digit (0‑9) in the first clause of the headline (before any period),0.0670,0.0000,0.2388,0.0000,0.1717,True,0.5223
3,"Mentions visual media (photos/graphics/images) in positive samples and mentions non‑visual tools or technologies (apps, ingredients, tattoos) in negative samples",0.0647,0.0000,0.5570,0.0000,0.1570,True,0.5234
